# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
import pandas as pd
import numpy as np

# Sample data (replace with your dataset if loaded)
df = pd.DataFrame({
    "page": ["Page A", "Page B", "Page C", "Page D", "Page E"],
    "ctr": [2.5, 7.2, 1.8, 4.3, 8.1],
    "average_position": [18, 6, 24, 14, 3],
    "impressions": [550, 120, 840, 210, 90]
})

def get_reason(row):
    reasons = []

    if row["ctr"] < 5:
        reasons.append("LOW_CTR")

    if row["average_position"] > 10:
        reasons.append("LOW_RANK")

    if row["impressions"] > 100:
        reasons.append("HIGH_IMPRESSIONS")

    if len(reasons) >= 2:
        reasons.append("REFRESH_CONTENT")

    return ", ".join(reasons)

df["reason_code"] = df.apply(get_reason, axis=1)

df

,page,ctr,average_position,impressions,reason_code
0,Page A,2.5,18,550,"LOW_CTR, LOW_RANK, HIGH_IMPRESSIONS, REFRESH_C..."
1,Page B,7.2,6,120,HIGH_IMPRESSIONS
2,Page C,1.8,24,840,"LOW_CTR, LOW_RANK, HIGH_IMPRESSIONS, REFRESH_C..."
3,Page D,4.3,14,210,"LOW_CTR, LOW_RANK, HIGH_IMPRESSIONS, REFRESH_C..."
4,Page E,8.1,3,90,


# My Rule

## Rule

If a page has:

- CTR less than 5%
- Average Position greater than 10
- More than 100 impressions

then the page should be reviewed and refreshed because it receives impressions but does not attract enough clicks.

## Reason Codes

| Code | Meaning |
|------|---------|
| LOW_CTR | Click-through rate is below 5% |
| LOW_RANK | Average position is greater than 10 |
| HIGH_IMPRESSIONS | Page receives more than 100 impressions |
| REFRESH_CONTENT | Recommend updating or improving the page |

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# Calculate a simple score

df["score"] = (
    df["impressions"] * 0.3
    + (100 - df["average_position"]) * 2
    + (5 - df["ctr"]).clip(lower=0) * 10
)

# Decide action
def get_action(row):
    if row["score"] >= 150:
        return "Refresh Content"
    elif row["score"] >= 100:
        return "Monitor"
    else:
        return "No Action"

df["action"] = df.apply(get_action, axis=1)

# Sort by score
ranked_df = df.sort_values(by="score", ascending=False)

# Show results
ranked_df

,page,ctr,average_position,impressions,reason_code,score,action
2,Page C,1.8,24,840,"LOW_CTR, LOW_RANK, HIGH_IMPRESSIONS, REFRESH_C...",436.0,Refresh Content
0,Page A,2.5,18,550,"LOW_CTR, LOW_RANK, HIGH_IMPRESSIONS, REFRESH_C...",354.0,Refresh Content
3,Page D,4.3,14,210,"LOW_CTR, LOW_RANK, HIGH_IMPRESSIONS, REFRESH_C...",242.0,Refresh Content
1,Page B,7.2,6,120,HIGH_IMPRESSIONS,224.0,Refresh Content
4,Page E,8.1,3,90,,221.0,Refresh Content


In [10]:
import os

os.makedirs("work/outputs", exist_ok=True)

ranked_df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("✅ CSV saved successfully!")

✅ CSV saved successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# Show the Top 20 pages (or fewer if your dataset has less)

# Ensure ranked_df is defined. If this cell is run out of order, it might not be.
# Please ensure all cells from Section 2 (Build the ranked queue) have been executed.
if 'ranked_df' not in locals() and 'ranked_df' not in globals():
    raise NameError("ranked_df is not defined. Please run the cells in Section 2 to create 'ranked_df' first.")

# Use .copy() to prevent SettingWithCopyWarning when adding new columns
top20 = ranked_df.head(20).copy()

# Add 'Confidence' column as requested
top20["Confidence"] = "Medium"

# Add 'What could make it wrong' column as requested
top20["What could make it wrong"] = (
    "Recent content updates or seasonal traffic changes, or changes in external factors like competitor activity."
)

# Display the selected columns for the top 20 pages
display(top20[[
    "page",
    "score",
    "action",
    "reason_code",
    "Confidence",
    "What could make it wrong"
]])

,page,score,action,reason_code,Confidence,What could make it wrong
2,Page C,436.0,Refresh Content,"LOW_CTR, LOW_RANK, HIGH_IMPRESSIONS, REFRESH_C...",Medium,Recent content updates or seasonal traffic cha...
0,Page A,354.0,Refresh Content,"LOW_CTR, LOW_RANK, HIGH_IMPRESSIONS, REFRESH_C...",Medium,Recent content updates or seasonal traffic cha...
3,Page D,242.0,Refresh Content,"LOW_CTR, LOW_RANK, HIGH_IMPRESSIONS, REFRESH_C...",Medium,Recent content updates or seasonal traffic cha...
1,Page B,224.0,Refresh Content,HIGH_IMPRESSIONS,Medium,Recent content updates or seasonal traffic cha...
4,Page E,221.0,Refresh Content,,Medium,Recent content updates or seasonal traffic cha...


### Summary of Top 20 Recommendations

Based on the scoring model, the top 20 pages represent those with the highest potential for improvement or impact. The `Action` column suggests whether to 'Refresh Content', 'Monitor', or take 'No Action'. The `Reason Code` provides insight into why a particular action is recommended, often highlighting low CTR, low rank, or high impressions.

The `Confidence` in these recommendations is set to 'Medium', acknowledging that the scoring model is a baseline. `What could make it wrong` highlights potential external factors or recent changes (like content updates or seasonal traffic) that the current model might not account for, suggesting areas for future refinement and manual review.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Explaining Weak Picks

'Weak picks' are recommendations that might not be as impactful or accurate as others, despite a high score. They could arise for several reasons:

1.  **Outdated Data**: If the underlying data (`df`) is old, the recommendations might not reflect current user behavior or content performance.
2.  **Generic Scoring**: Our current scoring formula is general. It doesn't account for nuances like page importance, content age, or specific business goals. For example, a page with moderate impressions but very high strategic value might be overlooked, while a high-impression, low-CTR page of lesser importance might be prioritized.
3.  **Edge Cases**: Some pages might have unusual combinations of metrics that result in a high score but are not genuinely problematic or highly optimizable. For instance, a very new page might have high impressions but low CTR simply because it hasn't had time to rank well or be discovered.
4.  **Assumptions in Reason Codes**: The `reason_code` logic might be too simplistic. A `LOW_CTR` might be intentional for certain types of content (e.g., informational pages not meant for direct conversion).
5.  **External Factors Not Modeled**: Trends, seasonality, competitor actions, or external events can heavily influence page performance, but our current model does not incorporate these, leading to potentially misleading recommendations.

### Leakage Check

Data leakage occurs when information that would not realistically be available at the time of prediction is used to create a model or make a recommendation. In the context of action recommendations for pages, this often means using 'future' metrics that reflect outcomes *after* an action would have been taken. Using such data can make a model appear highly accurate during development but perform poorly in real-world application because those 'future' signals are not available when making live recommendations.

In [12]:
# List of column names to check for potential data leakage
leakage_columns = [
    "future_clicks",
    "future_ctr",
    "future_impressions",
    "label",
    "target",
    "conversion",
    "future_rank"
]

# Check if any leakage columns are present in the dataframe
found_leakage_columns = [col for col in leakage_columns if col in df.columns]

if found_leakage_columns:
    print(f"⚠️ WARNING: Potential data leakage detected! The following columns were found in the dataframe: {', '.join(found_leakage_columns)}. These columns might contain future information that should not be used for current predictions or recommendations.")
else:
    print("✅ No leakage detected.")

✅ No leakage detected.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.